# 40 Train SFT Unsloth Qwen3.5 4B

Local planning/execution notebook for SFT.

Defaults are safe:
- BF16 LoRA only
- no QLoRA / no 4-bit training
- first 50-step sanity run enabled
- training execution disabled unless explicitly opted in


In [ ]:
from __future__ import annotations

import json
import platform
import sys
from pathlib import Path

import yaml

from lumis1.hashing import sha256_file
from lumis1.run_evidence import create_run_evidence_tree, write_run_status, write_run_summary

REPO_ROOT = Path.cwd().resolve()
REPORTS = REPO_ROOT / "workspace" / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)

train_cfg = yaml.safe_load((REPO_ROOT / "configs" / "train_sft.yaml").read_text(encoding="utf-8"))
profile_cfg = yaml.safe_load((REPO_ROOT / "configs" / "run_profiles.yaml").read_text(encoding="utf-8"))
chat_cfg = yaml.safe_load((REPO_ROOT / "configs" / "chat_template_policy.yaml").read_text(encoding="utf-8"))

PROFILE = "default_96gb"
RUN_ID = "manual-sft"
RUN_TRAINING = False
FIRST_50_STEPS_SANITY = True

if train_cfg["model"]["dtype"].lower() != "bf16":
    raise RuntimeError("SFT must use BF16")
if train_cfg["model"].get("load_in_4bit") is not False:
    raise RuntimeError("QLoRA / 4-bit training is disabled for Qwen3.5")
if chat_cfg["allowed_template"]["expected_chat_template_kwargs"]["enable_thinking"] is not False:
    raise RuntimeError("thinking must be disabled by policy")

profile = profile_cfg["profiles"][PROFILE]["sft"]
SFT_OUTPUT_DIR = REPO_ROOT / train_cfg["outputs"]["run_dir"]
resolved = {
    "profile": PROFILE,
    "run_id": RUN_ID,
    "model": train_cfg["model"],
    "lora": train_cfg["lora"],
    "training": dict(train_cfg["training"], **profile),
    "chat_template_policy": chat_cfg,
    "dataset_path": str(REPO_ROOT / train_cfg["datasets"]["train_sft_path"]),
    "run_training": RUN_TRAINING,
    "output_dir": str(SFT_OUTPUT_DIR),
}
if FIRST_50_STEPS_SANITY:
    resolved["training"]["max_steps"] = int(train_cfg["sanity_run"]["max_steps"])

out = REPO_ROOT / train_cfg["outputs"]["resolved_config_report"]
out.write_text(json.dumps(resolved, indent=2), encoding="utf-8")

run_paths = None
if RUN_TRAINING:
    run_paths = create_run_evidence_tree(REPO_ROOT, RUN_ID)
    (run_paths["config_snapshot"] / "train_sft_resolved.json").write_text(
        json.dumps(resolved, indent=2),
        encoding="utf-8",
    )
    (run_paths["commands"] / "notebook_40_invocation.txt").write_text(
        "Notebook 40 manual execution for SFT training.\n"
        f"profile={PROFILE}\n"
        f"output_dir={SFT_OUTPUT_DIR}\n",
        encoding="utf-8",
    )
    (run_paths["environment"] / "runtime.json").write_text(
        json.dumps(
            {
                "python": sys.version,
                "platform": platform.platform(),
                "cwd": str(REPO_ROOT),
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    write_run_status(
        REPO_ROOT,
        RUN_ID,
        stage="sft",
        status="running",
        details={"profile": PROFILE, "output_dir": str(SFT_OUTPUT_DIR)},
    )

print(json.dumps(resolved, indent=2))
print(f"saved: {out}")


In [ ]:
# Optional execution block.
# This block is intentionally gated to avoid accidental local training runs.
# Set RUN_TRAINING = True in the previous cell before executing this cell.

if not RUN_TRAINING:
    print("RUN_TRAINING is False. No training executed.")
else:
    import torch
    from datasets import load_dataset
    from trl import SFTConfig, SFTTrainer
    from unsloth import FastLanguageModel

    try:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=train_cfg["model"]["base_model"],
            max_seq_length=int(resolved["training"].get("max_seq_length", 4096)),
            load_in_4bit=False,
            dtype=torch.bfloat16,
        )
        model = FastLanguageModel.get_peft_model(
            model,
            r=int(train_cfg["lora"]["r"]),
            lora_alpha=int(train_cfg["lora"]["lora_alpha"]),
            lora_dropout=float(train_cfg["lora"]["lora_dropout"]),
            target_modules=train_cfg["lora"]["target_modules"],
            bias=train_cfg["lora"]["bias"],
        )

        ds = load_dataset("json", data_files=resolved["dataset_path"], split="train")
        trainer = SFTTrainer(
            model=model,
            tokenizer=tokenizer,
            train_dataset=ds,
            args=SFTConfig(**resolved["training"], output_dir=str(SFT_OUTPUT_DIR)),
        )
        train_result = trainer.train()
        trainer.save_model(str(SFT_OUTPUT_DIR))

        train_metrics = getattr(train_result, "metrics", {})
        checksums = {
            str(path.relative_to(SFT_OUTPUT_DIR)): sha256_file(path)
            for path in sorted(SFT_OUTPUT_DIR.rglob("*"))
            if path.is_file()
        }
        report = {
            "run_id": RUN_ID,
            "output_dir": str(SFT_OUTPUT_DIR),
            "metrics": train_metrics,
            "artifact_files": sorted(checksums),
        }
        if run_paths is not None:
            (run_paths["reports"] / "sft_training.json").write_text(
                json.dumps(report, indent=2),
                encoding="utf-8",
            )
            (run_paths["checksums"] / "artifacts.json").write_text(
                json.dumps(checksums, indent=2),
                encoding="utf-8",
            )
            write_run_status(REPO_ROOT, RUN_ID, stage="sft", status="completed", details=report)
            write_run_summary(
                REPO_ROOT,
                RUN_ID,
                "# SFT Summary\n\nThis summary reflects notebook 40 execution.\n\n```json\n"
                + json.dumps(report, indent=2)
                + "\n```",
            )
        print(json.dumps(report, indent=2))
    except Exception as exc:
        if run_paths is not None:
            write_run_status(
                REPO_ROOT,
                RUN_ID,
                stage="sft",
                status="failed",
                details={"error": str(exc), "output_dir": str(SFT_OUTPUT_DIR)},
            )
        raise
